In [1]:
import arcpy
import os
arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = None
project_folder = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_Sackett_Impacts/Wetland Mapping"
wetland_gdb = os.path.normpath(os.path.join(project_folder, "Illinois_Wetland_Shapefiles.gdb"))
waters_gdb = os.path.normpath(os.path.join(project_folder, "Jurisdictional_Water_Shapefiles.gdb"))

# Intersect NWI with NLD leveed areas layer

In [2]:
# NLCD years
years = [2023, 2024]
n_y = len(years)

In [3]:
# all wetlands above area threshold
step11 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step11_{y}NLCD_AreaThreshold")) for y in years]

In [4]:
# path to NLD leveed areas layer
leveed_areas = os.path.normpath(os.path.join(project_folder, r"Wetland Mapping.gdb/LeveedAreas_Clip_AlbersNAD1983"))
levee_field = "Within_Lev"

In [5]:
for i in range(n_y):
    # create levee intersect field
    if not arcpy.ListFields(step11[i], levee_field):
        arcpy.management.AddField(step11[i], levee_field, "LONG");

    # pre-fill field with zeros
    arcpy.management.CalculateField(
        in_table=step11[i],
        field=levee_field,
        expression="0",
        expression_type="PYTHON3"
    );

    # select all wetland polygons that occur within leveed area
    arcpy.management.MakeFeatureLayer(step11[i], "temp_layer");
    arcpy.management.SelectLayerByLocation(
        in_layer="temp_layer",
        overlap_type="WITHIN",
        select_features=leveed_areas,
        selection_type="NEW_SELECTION"
    );
    count = int(arcpy.management.GetCount("temp_layer")[0])
    print(f"Selected: {count} features")

    # calculate 1 only for selected features
    arcpy.management.CalculateField(
        in_table="temp_layer",
        field=levee_field,
        expression="1",
        expression_type="PYTHON3"
    );
    arcpy.management.Delete("temp_layer");

Selected: 7469 features
Selected: 7506 features


# Intersect NWI with jurisdictional waters layers

In [19]:
# buffer distances
buffers = [1, 10, 20, 100]
n_b = len(buffers)

In [20]:
# flow permanence levels
permanence_levels = [1, 2, 3]
n_p = len(permanence_levels)

In [21]:
# create old NHD fields for each flow permanence and buffer distance combination
nhd_fields = [f"NHDIn{l}_{b}" for l in permanence_levels for b in buffers]
nhd_layers = [f"NHD_Jurisdictional_Waters_{l}_{b}" for l in permanence_levels for b in buffers]
nhd_paths = [os.path.normpath(os.path.join(waters_gdb, f"{layer}")) for layer in nhd_layers]

In [22]:
# create old Brinkerhoff fields for each flow permanence and buffer distance combination
brink_fields = [f"BRFIn{l}_{b}" for l in permanence_levels for b in buffers]
brink_layers = [f"Brinkerhoff_Jurisdictional_Waters_{l}_{b}" for l in permanence_levels for b in buffers]
brink_paths = [os.path.normpath(os.path.join(waters_gdb, f"{layer}")) for layer in brink_layers]

In [25]:
# NHD: create new fields and fill with zeros
for i in range(n_y):
    for f in nhd_fields:
        existing = [f.name for f in arcpy.ListFields(step11[i])]
        if f not in existing:
            arcpy.management.AddField(step11[i], f, "LONG");            
            arcpy.management.CalculateField(
                in_table=step11[i],
                field=f,
                expression="0",
                expression_type="PYTHON3"
            );

In [28]:
# Brinkerhoff: create new fields and fill with zeros
for i in range(n_y):
    for f in brink_fields:
        existing = [f.name for f in arcpy.ListFields(step11[i])]
        if f not in existing:
            arcpy.management.AddField(step11[i], f, "LONG");      
            arcpy.management.CalculateField(
                in_table=step11[i],
                field=f,
                expression="0",
                expression_type="PYTHON3"
            );

In [29]:
# NHD: set field values to 1 if they intersect with jurisdictional waters
for i in range(n_y):
    for j in range(len(nhd_fields)):
        arcpy.management.MakeFeatureLayer(step11[i], "temp_layer");
        arcpy.management.SelectLayerByLocation(
            in_layer="temp_layer",
            overlap_type="INTERSECT",
            select_features=nhd_paths[j],
            selection_type="NEW_SELECTION"
        );
        arcpy.management.CalculateField(
            in_table="temp_layer", 
            field=nhd_fields[j],
            expression="1",
            expression_type="PYTHON3"
        );
        arcpy.management.Delete("temp_layer");

In [30]:
for i in range(n_y):
    for j in range(len(brink_fields)):
        arcpy.management.MakeFeatureLayer(step11[i], "temp_layer");
        arcpy.management.SelectLayerByLocation(
            in_layer="temp_layer",
            overlap_type="INTERSECT",
            select_features=brink_paths[j],
            selection_type="NEW_SELECTION"
        );
        arcpy.management.CalculateField(
            in_table="temp_layer",      # calculates on selection only
            field=brink_fields[j],
            expression="1",
            expression_type="PYTHON3"
        );
        arcpy.management.Delete("temp_layer");